# LeetCode 800 — Similar RGB Color bằng Python

> **Bài Premium: đề bài trong notebook được viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.**

**Link đề:** https://leetcode.com/problems/similar-rgb-color/
**Độ khó:** Easy · **Chủ đề:** Math, String, Enumeration (liệt kê)

```python
class Solution:
    def similarRGB(self, color: str) -> str:
        ...
```

## Mục tiêu học
1. Hiểu **hệ thập lục phân** (hexadecimal, cơ số 16) và mã màu `#RRGGBB`.
2. Dùng thành thạo `int(s, 16)`, `format(x, "02x")`, f-string `{x:02x}` để đổi qua lại hex ↔ số.
3. Nhận ra màu "viết tắt được" có mỗi thành phần là **bội số của 17** (`0x00, 0x11, ..., 0xff`).
4. Đi từ liệt kê toàn bộ `16³ = 4096` màu → tách độc lập từng thành phần → **làm tròn** `round(v / 17)` trong `O(1)`.
5. Trace, chứng minh tách được từng thành phần, phân tích Time/Space, luyện biến thể.

**Cách học:** đọc markdown → đoán output → chạy → so sánh. Cell TODO không tự chạy.

## 1. Đề bài (viết theo hiểu biết)

Mỗi chữ in hoa dưới đây đại diện cho **một chữ số hex** `0..f`.

- Màu `"#AABBCC"` có thể **viết tắt** thành `"#ABC"`. Ví dụ `"#15c"` là viết tắt của `"#1155cc"`.
- **Độ tương đồng** giữa hai màu `"#ABCDEF"` và `"#UVWXYZ"` là:
  `-(AB - UV)² - (CD - WX)² - (EF - YZ)²` — trong đó `AB`, `UV`, ... là các số hex 2 chữ số (0..255).

Cho chuỗi `color` dạng `"#ABCDEF"`, trả về chuỗi màu dạng **7 ký tự** `"#XXYYZZ"` **có thể viết tắt** và **giống `color` nhất** (độ tương đồng lớn nhất, tức tổng bình phương sai lệch nhỏ nhất). Nếu có nhiều đáp án cùng độ tương đồng cao nhất thì trả về đáp án nào cũng được.

**Ví dụ 1:** `color = "#09f166"` → `"#11ee66"`
Độ tương đồng: `-(0x09 - 0x11)² - (0xf1 - 0xee)² - (0x66 - 0x66)² = -64 - 9 - 0 = -73`, lớn nhất trong mọi màu viết tắt được.

**Ví dụ 2:** `color = "#4e3fe1"` → `"#5544dd"`

**Constraints (theo hiểu biết):** `color.length == 7`, `color[0] == '#'`, các ký tự còn lại là chữ số hoặc chữ thường `'a'..'f'`.

## 2. Nền tảng: hệ thập lục phân (hex)

Hệ hex có 16 chữ số: `0 1 2 3 4 5 6 7 8 9 a b c d e f` (`a=10, ..., f=15`). Mỗi vị trí mạnh gấp **16** lần vị trí bên phải.

Số hex 2 chữ số `XY` có giá trị `X·16 + Y`, chạy từ `0x00 = 0` đến `0xff = 255`.

| hex | tính | thập phân |
|---|---|---|
| `09` | 0·16 + 9 | 9 |
| `f1` | 15·16 + 1 | 241 |
| `11` | 1·16 + 1 | 17 |
| `ee` | 14·16 + 14 | 238 |

Mã màu `#RRGGBB` gồm 3 thành phần đỏ (R), xanh lá (G), xanh dương (B), mỗi thành phần 0..255.

In [1]:
print(int("09", 16), int("f1", 16), int("11", 16), int("ee", 16))
print(0x66, 0xff)                 # viết hằng số hex trực tiếp trong Python bằng tiền tố 0x
print(hex(238), "| format(238, '02x') =", format(238, "02x"), "| format(5, '02x') =", format(5, "02x"))
print(f"{17:02x} {0:02x} {255:02x}")

9 241 17 238
102 255
0xee | format(238, '02x') = ee | format(5, '02x') = 05
11 00 ff


### 2.1. Đọc các công cụ vừa dùng

- `int(s, 16)`: đọc chuỗi `s` theo cơ số 16 → số nguyên.
- `hex(x)`: số → chuỗi có tiền tố `'0x'` (`'0xee'`). Ít dùng cho bài này vì có `0x` và không đệm số 0.
- `format(x, "02x")` hoặc `f"{x:02x}"`: in `x` dạng hex chữ thường (`x`), **độ rộng 2**, đệm `0` bên trái. Đệm rất quan trọng: `5` phải thành `"05"`, không phải `"5"`.

### 2.2. Cắt chuỗi màu thành 3 thành phần

`color = "#09f166"` → `color[1:3] = "09"`, `color[3:5] = "f1"`, `color[5:7] = "66"`. Dùng `range(1, 7, 2)` để sinh vị trí bắt đầu 1, 3, 5 (bước nhảy 2).

In [2]:
color = "#09f166"
print(list(range(1, 7, 2)))
parts = [color[i:i + 2] for i in range(1, 7, 2)]
print(parts, "->", [int(p, 16) for p in parts])

[1, 3, 5]
['09', 'f1', '66'] -> [9, 241, 102]


## 3. Nền tảng tư duy: màu viết tắt được = bội số của 17

Thành phần viết tắt được có dạng `XX` (hai chữ số hex **giống nhau**). Giá trị:

`XX = X·16 + X = 17·X`, với `X = 0..15`.

→ Các giá trị hợp lệ: `0, 17, 34, ..., 255` — đúng **16 bội số của 17**. Cách nhau đều 17 đơn vị.

In [3]:
valid = [17 * x for x in range(16)]
print(valid)
print([format(v, "02x") for v in valid])

[0, 17, 34, 51, 68, 85, 102, 119, 136, 153, 170, 187, 204, 221, 238, 255]
['00', '11', '22', '33', '44', '55', '66', '77', '88', '99', 'aa', 'bb', 'cc', 'dd', 'ee', 'ff']


### 3.1. Hình dung: một thành phần nằm giữa hai "vạch" bội số của 17

Lấy `v = 0xf1 = 241`. Hai vạch kề nó là `238 (ee)` và `255 (ff)`. Khoảng cách tương ứng 3 và 14 → vạch `ee` gần hơn. Cell dưới in khoảng cách từ `v` tới **mọi** vạch, ta thấy khoảng cách giảm dần rồi tăng dần (dạng chữ V) — điểm thấp nhất là vạch gần nhất.

In [4]:
v = 0xf1
for x in range(16):
    d = abs(17 * x - v)
    print(f"{17*x:02x} ({17*x:>3}) | khoảng cách {d:>3} " + "#" * (d // 8))

00 (  0) | khoảng cách 241 ##############################
11 ( 17) | khoảng cách 224 ############################
22 ( 34) | khoảng cách 207 #########################
33 ( 51) | khoảng cách 190 #######################
44 ( 68) | khoảng cách 173 #####################
55 ( 85) | khoảng cách 156 ###################
66 (102) | khoảng cách 139 #################
77 (119) | khoảng cách 122 ###############
88 (136) | khoảng cách 105 #############
99 (153) | khoảng cách  88 ###########
aa (170) | khoảng cách  71 ########
bb (187) | khoảng cách  54 ######
cc (204) | khoảng cách  37 ####
dd (221) | khoảng cách  20 ##
ee (238) | khoảng cách   3 
ff (255) | khoảng cách  14 #


## 4. Cách 1 — Liệt kê toàn bộ 16³ = 4096 màu viết tắt (brute force)

Thử mọi bộ `(X, Y, Z)` với `X, Y, Z ∈ 0..15`, tính độ tương đồng với `color`, giữ màu tốt nhất. Đúng theo định nghĩa đề, rất dễ kiểm chứng.

In [5]:
def similarity(c1: str, c2: str) -> int:
    total = 0
    for i in range(1, 7, 2):
        diff = int(c1[i:i + 2], 16) - int(c2[i:i + 2], 16)
        total -= diff * diff
    return total

print("similarity('#09f166', '#11ee66') =", similarity("#09f166", "#11ee66"))

def similar_rgb_bruteforce(color: str) -> str:
    best, best_sim = None, None
    for x in range(16):
        for y in range(16):
            for z in range(16):
                cand = "#" + format(17 * x, "02x") + format(17 * y, "02x") + format(17 * z, "02x")
                sim = similarity(color, cand)
                if best_sim is None or sim > best_sim:
                    best, best_sim = cand, sim
    return best

print(similar_rgb_bruteforce("#09f166"))   # #11ee66
print(similar_rgb_bruteforce("#4e3fe1"))   # #5544dd

similarity('#09f166', '#11ee66') = -73
#11ee66
#5544dd


**Độ phức tạp:** 4096 ứng viên × 3 thành phần → hằng số (`O(1)` theo nghĩa input cố định 7 ký tự), nhưng là ~12 000 phép tính — lãng phí.

## 5. Nút thắt và insight — ba thành phần **độc lập** nhau

Độ tương đồng là **tổng** ba số hạng, mỗi số hạng chỉ phụ thuộc **một** thành phần:

`-(R - R')² - (G - G')² - (B - B')²`

Muốn tổng lớn nhất ⇒ làm **từng số hạng** lớn nhất riêng rẽ ⇒ với mỗi thành phần, chọn bội số của 17 **gần nhất**. Từ `16³` ứng viên còn `3 × 16` lần thử.

In [6]:
def closest_component_scan(hex2: str) -> str:
    v = int(hex2, 16)
    best = min(range(16), key=lambda x: abs(17 * x - v))   # X làm |17X - v| nhỏ nhất
    return format(17 * best, "02x")

for p in ["09", "f1", "66", "4e", "3f", "e1"]:
    print(p, "=", int(p, 16), "-> gần nhất", closest_component_scan(p), "=", int(closest_component_scan(p), 16))

09 = 9 -> gần nhất 11 = 17
f1 = 241 -> gần nhất ee = 238
66 = 102 -> gần nhất 66 = 102
4e = 78 -> gần nhất 55 = 85
3f = 63 -> gần nhất 44 = 68
e1 = 225 -> gần nhất dd = 221


### 5.1. `min(iterable, key=...)` — lấy phần tử có "điểm" nhỏ nhất

`min(range(16), key=f)` duyệt `x = 0..15`, tính `f(x)` và trả về **chính `x`** (không phải `f(x)`) có `f(x)` nhỏ nhất. Nếu hòa, lấy phần tử **gặp trước**.

In [7]:
words = ["banana", "kiwi", "apple"]
print(min(words, key=len))            # 'kiwi' (độ dài 4)
print(min([3, -1, 2], key=abs))       # -1 (|−1| nhỏ nhất)

kiwi
-1


## 6. Lời giải chính — làm tròn `round(v / 17)` trong `O(1)`

Bội số của 17 gần `v` nhất là `17 · q` với `q` = `v / 17` **làm tròn tới số nguyên gần nhất**.

Để tránh số thực, dùng công thức số nguyên: `q = (v + 8) // 17`.
- Vì sao `+8`? Làm tròn `v/17` ⇔ `floor((v + 17/2) / 17)`; `17/2 = 8.5`. Phần dư `v % 17` ∈ `0..16`: dư `≤ 8` thì làm tròn xuống, `≥ 9` làm tròn lên. `(v + 8) // 17` làm đúng như vậy.
- **Không bao giờ hòa**: khoảng cách tới hai bội liền kề là `r` và `17 - r`; bằng nhau cần `r = 8.5` — không phải số nguyên.

Ví dụ `v = 0x09 = 9`: `(9 + 8) // 17 = 1` → `17` = `0x11`. `v = 0xf1 = 241`: `(241 + 8) // 17 = 14` → `238` = `0xee`.

In [8]:
for v in [0, 8, 9, 16, 17, 25, 26, 241, 255]:
    q = (v + 8) // 17
    print(f"v={v:>3} | v%17={v % 17:>2} | q={q:>2} | 17q={17*q:>3} = 0x{17*q:02x}")

v=  0 | v%17= 0 | q= 0 | 17q=  0 = 0x00
v=  8 | v%17= 8 | q= 0 | 17q=  0 = 0x00
v=  9 | v%17= 9 | q= 1 | 17q= 17 = 0x11
v= 16 | v%17=16 | q= 1 | 17q= 17 = 0x11
v= 17 | v%17= 0 | q= 1 | 17q= 17 = 0x11
v= 25 | v%17= 8 | q= 1 | 17q= 17 = 0x11
v= 26 | v%17= 9 | q= 2 | 17q= 34 = 0x22
v=241 | v%17= 3 | q=14 | 17q=238 = 0xee
v=255 | v%17= 0 | q=15 | 17q=255 = 0xff


In [9]:
def similar_rgb(color: str) -> str:
    result = "#"
    for i in range(1, 7, 2):
        v = int(color[i:i + 2], 16)      # (1) đọc thành phần
        q = (v + 8) // 17                # (2) chỉ số bội gần nhất 0..15
        result += f"{17 * q:02x}"        # (3) ghi lại dạng hex 2 chữ số
    return result

print(similar_rgb("#09f166"), similar_rgb("#4e3fe1"))

#11ee66 #5544dd


## 7. TRACE từng bước

`color = "#09f166"`:

| i | hex | v | v % 17 | q = (v+8)//17 | 17q | hex kết quả | result sau |
|---|---|---|---|---|---|---|---|
| 1 | 09 | 9 | 9 | 1 | 17 | 11 | `#11` |
| 3 | f1 | 241 | 3 | 14 | 238 | ee | `#11ee` |
| 5 | 66 | 102 | 0 | 6 | 102 | 66 | `#11ee66` |

In [10]:
def trace_similar_rgb(color: str) -> str:
    print(f"color = {color}")
    result = "#"
    for i in range(1, 7, 2):
        h = color[i:i + 2]
        v = int(h, 16)
        q = (v + 8) // 17
        new_h = f"{17 * q:02x}"
        print(f"  i={i}: '{h}' = {v:>3} | dư {v % 17:>2} -> q={q:>2} -> {17*q:>3} = '{new_h}' "
              f"| sai lệch {v - 17*q:+d}")
        result += new_h
    print("  kết quả:", result, "| similarity =", similarity(color, result))
    return result

_ = trace_similar_rgb("#09f166")
_ = trace_similar_rgb("#4e3fe1")

color = #09f166
  i=1: '09' =   9 | dư  9 -> q= 1 ->  17 = '11' | sai lệch -8
  i=3: 'f1' = 241 | dư  3 -> q=14 -> 238 = 'ee' | sai lệch +3
  i=5: '66' = 102 | dư  0 -> q= 6 -> 102 = '66' | sai lệch +0
  kết quả: #11ee66 | similarity = -73
color = #4e3fe1
  i=1: '4e' =  78 | dư 10 -> q= 5 ->  85 = '55' | sai lệch -7
  i=3: '3f' =  63 | dư 12 -> q= 4 ->  68 = '44' | sai lệch -5
  i=5: 'e1' = 225 | dư  4 -> q=13 -> 221 = 'dd' | sai lệch +4
  kết quả: #5544dd | similarity = -90


`{x:+d}` in số nguyên **kèm dấu** (`+3`, `-8`), giúp thấy làm tròn lên hay xuống.

## 8. Code nộp LeetCode — lời giải chính

In [11]:
class Solution:
    def similarRGB(self, color: str) -> str:
        return "#" + "".join(f"{17 * ((int(color[i:i + 2], 16) + 8) // 17):02x}" for i in range(1, 7, 2))

sol = Solution()
assert sol.similarRGB("#09f166") == "#11ee66"
assert sol.similarRGB("#4e3fe1") == "#5544dd"
print("Solution OK")

Solution OK


### `"".join(...)` — nối nhiều chuỗi

`sep.join(iterable)` nối các chuỗi với `sep` ở giữa. `"".join(["11","ee","66"]) = "11ee66"`. Nhanh và gọn hơn cộng chuỗi trong vòng lặp.

In [12]:
print("".join(["11", "ee", "66"]), "-".join(["a", "b", "c"]))

11ee66 a-b-c


## 9. Cách 4 — Chỉ nhìn chữ số hex (không cần đổi cả số)

Với thành phần `"AB"` (chữ số đầu `A`, chữ số sau `B`): ứng viên gần nhất chỉ có thể là `AA`, `(A-1)(A-1)` hoặc `(A+1)(A+1)`. Ta thử 3 ứng viên này (trong khoảng 0..15) và chọn cái gần nhất. Cách này hay gặp trong lời giải "suy luận theo chữ số".

In [13]:
HEX = "0123456789abcdef"

def closest_by_digits(h: str) -> str:
    v = int(h, 16)
    a = int(h[0], 16)
    cands = [d for d in (a - 1, a, a + 1) if 0 <= d <= 15]
    best = min(cands, key=lambda d: abs(17 * d - v))
    return HEX[best] * 2                     # "e" * 2 = "ee"

def similar_rgb_digits(color: str) -> str:
    return "#" + "".join(closest_by_digits(color[i:i + 2]) for i in range(1, 7, 2))

print("e" * 2, "|", similar_rgb_digits("#09f166"), similar_rgb_digits("#4e3fe1"))

ee | #11ee66 #5544dd


In [14]:
class Solution:
    def similarRGB(self, color: str) -> str:
        hexd = "0123456789abcdef"
        out = "#"
        for i in range(1, 7, 2):
            v = int(color[i:i + 2], 16)
            q = round(v / 17)          # round() của Python: banker's rounding khi .5, nhưng ở đây không bao giờ .5
            out += hexd[q] * 2
        return out

print(Solution().similarRGB("#09f166"), Solution().similarRGB("#4e3fe1"))

#11ee66 #5544dd


## 10. Vì sao đúng?

**Tách biến:** với mọi màu viết tắt `(R', G', B')`, `sim = -(R-R')² - (G-G')² - (B-B')²`. Các số hạng không ràng buộc nhau (chọn `R'` không ảnh hưởng lựa chọn `G'`, `B'`), nên max của tổng = tổng các max riêng.

**Làm tròn:** với `v = 17k + r`, `0 <= r <= 16`, hai bội gần nhất là `17k` (cách `r`) và `17(k+1)` (cách `17 - r`). Chọn `17k` ⇔ `r <= 8` ⇔ `(v + 8) // 17 = k`. Chọn `17(k+1)` ⇔ `r >= 9` ⇔ `(v + 8) // 17 = k + 1`. Không vượt biên: `v <= 255 = 17·15` ⇒ `q <= 15`.

**Không có hòa** (`r = 8.5` là không thể) ⇒ đáp án là **duy nhất**, nên mọi cách đều phải ra cùng chuỗi.

## 11. Độ phức tạp và so sánh

Input luôn 7 ký tự → mọi cách đều `O(1)`; so sánh theo **số phép thử**:

| Cách | Số ứng viên thử | Time | Space | Ghi chú |
|---|---|---|---|---|
| 1. Liệt kê 4096 màu | 4096 | `O(16³)` | `O(1)` | Đúng định nghĩa, để kiểm chứng |
| 2. Tách thành phần + quét 16 | 3·16 | `O(3·16)` | `O(1)` | Dùng insight độc lập |
| 3. `(v + 8) // 17` | 3 | `O(1)` | `O(1)` | **Khuyến nghị** |
| 4. Chỉ xét 3 chữ số lân cận | 3·3 | `O(1)` | `O(1)` | Suy luận theo chữ số |

Nếu tổng quát hóa thành `k` thành phần, brute force là `16^k` (mũ!) còn cách tách là `O(k)` — đây là giá trị thật của insight.

## 12. Bộ test

1. Hai ví dụ.
2. Đã viết tắt được sẵn (`#000000`, `#ffffff`, `#aabbcc`) → giữ nguyên.
3. Ranh giới làm tròn: dư 8 (làm tròn xuống) và dư 9 (làm tròn lên): `0x08 = 8 → 00`, `0x09 = 9 → 11`; `0x19 = 25 = 17 + 8 → 11`, `0x1a = 26 = 17 + 9 → 22`.
4. Biên trên: `0xf7 = 247 → ee hay ff?` (247 = 238 + 9 → lên `ff`).
5. Đối chiếu **tất cả** 256 giá trị một thành phần và nhiều màu ngẫu nhiên với brute force.

In [15]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

check("đề 1", similar_rgb("#09f166"), "#11ee66")
check("đề 2", similar_rgb("#4e3fe1"), "#5544dd")
check("đen", similar_rgb("#000000"), "#000000")
check("trắng", similar_rgb("#ffffff"), "#ffffff")
check("đã viết tắt", similar_rgb("#aabbcc"), "#aabbcc")
check("dư 8 / 9", similar_rgb("#080900"), "#001100")
check("25 / 26", similar_rgb("#191a00"), "#112200")
check("247 -> ff", similar_rgb("#f7f6f5"), "#ffeeee")

PASS đề 1: #11ee66
PASS đề 2: #5544dd
PASS đen: #000000
PASS trắng: #ffffff
PASS đã viết tắt: #aabbcc
PASS dư 8 / 9: #001100
PASS 25 / 26: #112200
PASS 247 -> ff: #ffeeee


In [16]:
import random
random.seed(800)
# mọi giá trị của một thành phần
for v in range(256):
    c = "#" + f"{v:02x}" * 3
    assert similar_rgb(c) == similar_rgb_bruteforce(c) == similar_rgb_digits(c) == Solution().similarRGB(c)
# màu ngẫu nhiên
for _ in range(100):
    c = "#" + "".join(random.choice(HEX) for _ in range(6))
    expected = similar_rgb_bruteforce(c)
    assert similar_rgb(c) == expected and similar_rgb_digits(c) == expected and Solution().similarRGB(c) == expected
print("256 giá trị thành phần + 100 màu ngẫu nhiên: các cách khớp nhau")

256 giá trị thành phần + 100 màu ngẫu nhiên: các cách khớp nhau


## 13. Lỗi người mới hay gặp

### Lỗi 1: Quên đệm số 0
```python
result += hex(17 * q)[2:]     # SAI: 0 -> "0" (1 ký tự), chuỗi kết quả thiếu độ dài
```
Dùng `f"{x:02x}"`.

### Lỗi 2: Làm tròn xuống (`v // 17`) thay vì làm tròn gần nhất
`0x09 = 9 // 17 = 0` → `00` (lệch 9) trong khi `11` chỉ lệch 8.

### Lỗi 3: Chỉ lấy chữ số đầu nhân đôi (`"AB" → "AA"`)
`"f1" → "ff"` lệch 14, trong khi `"ee"` chỉ lệch 3.

### Lỗi 4: Trả chuỗi viết tắt 4 ký tự `"#1e6"`
Đề yêu cầu dạng đầy đủ 7 ký tự `"#11ee66"`.

In [17]:
def wrong_no_pad(color):
    return "#" + "".join(hex(17 * ((int(color[i:i+2], 16) + 8) // 17))[2:] for i in range(1, 7, 2))

def wrong_floor(color):
    return "#" + "".join(f"{17 * (int(color[i:i+2], 16) // 17):02x}" for i in range(1, 7, 2))

def wrong_first_digit(color):
    return "#" + "".join(color[i] * 2 for i in range(1, 7, 2))

c = "#09f166"
print("đúng          :", similar_rgb(c), similarity(c, similar_rgb(c)))
print("floor         :", wrong_floor(c), similarity(c, wrong_floor(c)))
print("chữ số đầu    :", wrong_first_digit(c), similarity(c, wrong_first_digit(c)))
print("không đệm 0   :", wrong_no_pad("#020304"), "(độ dài", len(wrong_no_pad("#020304")), ")")

đúng          : #11ee66 -73
floor         : #00ee66 -90
chữ số đầu    : #00ff66 -277
không đệm 0   : #000 (độ dài 4 )


## 14. Mẹo nhận dạng pattern

- Hàm mục tiêu là **tổng** các số hạng, mỗi số hạng phụ thuộc một biến riêng ⇒ **tối ưu từng biến độc lập** (separable).
- "Gần nhất trong một lưới đều bước `d`" ⇒ **làm tròn**: `round(v / d) * d` hoặc `(v + d // 2) // d * d` (cẩn thận khi `d` chẵn có thể hòa).
- Đề về màu / mã hex ⇒ `int(s, 16)`, `f"{x:02x}"`.
- Không gian tìm kiếm nhỏ (4096) ⇒ brute force là **công cụ kiểm chứng** tuyệt vời cho lời giải thông minh.

### Biến thể

1. Màu viết tắt **xa nhất** (similarity nhỏ nhất): mỗi thành phần chọn `00` hoặc `ff` tùy `v` gần đầu nào hơn.
2. Đổi màu sang **ảnh xám** (grayscale) gần nhất `#XXXXXX` (cả ba thành phần bằng nhau) — không còn độc lập; tối ưu một biến `X` với hàm bậc hai → trung bình cộng rồi làm tròn.
3. Làm tròn về lưới bước `d` bất kỳ (ví dụ bảng màu "web-safe" bước 51 = `0x33`).
4. Chuyển `"#15c"` (viết tắt) ↔ `"#1155cc"` (đầy đủ).

In [18]:
# Biến thể 3: bảng màu web-safe (mỗi thành phần là bội của 51: 00, 33, 66, 99, cc, ff)
def nearest_web_safe(color):
    out = "#"
    for i in range(1, 7, 2):
        v = int(color[i:i + 2], 16)
        q = min(range(6), key=lambda k: abs(51 * k - v))
        out += f"{51 * q:02x}"
    return out

print(nearest_web_safe("#09f166"), nearest_web_safe("#4e3fe1"))

#00ff66 #6633cc


## 15. Bài tập tự luyện (TODO — đáp án ở cuối)

**Bài 1 (tính tay):** Không chạy code, tính `similarRGB("#a7c3e0")`. Gán `answer_ex1`.

**Bài 2 (điền chỗ trống):** Hoàn thiện `nearest_component(h)` nhận chuỗi hex 2 ký tự, trả chuỗi `"XX"` gần nhất bằng công thức `(v + 8) // 17`.

**Bài 3 (chuyển đổi):** `expand_shorthand(short)` đổi `"#15c"` → `"#1155cc"`, và `to_shorthand(full)` đổi `"#1155cc"` → `"#15c"` (giả sử đầu vào viết tắt được).

**Bài 4 (biến thể):** `farthest_shorthand(color)` trả màu viết tắt **ít** giống `color` nhất (giả sử không có hòa: mọi thành phần khác `0x7f`/`0x80`... hãy tự lý giải khi nào hòa).

**Bài 5 (mini challenge):** `nearest_gray(color)` trả màu xám `#XXXXXX` (ba thành phần bằng nhau, `X` là số 0..255 bất kỳ, **không** cần viết tắt được) gần `color` nhất theo tổng bình phương.

In [19]:
answer_ex1 = None   # TODO: chuỗi 7 ký tự

def nearest_component(h: str) -> str:
    # TODO: v = int(h, 16); q = (v + 8) // 17; trả f"{17*q:02x}"
    pass

def expand_shorthand(short: str) -> str:
    # TODO
    pass

def to_shorthand(full: str) -> str:
    # TODO
    pass

# assert nearest_component("f1") == "ee"
# assert expand_shorthand("#15c") == "#1155cc" and to_shorthand("#1155cc") == "#15c"

In [20]:
def farthest_shorthand(color: str) -> str:
    # TODO
    pass

def nearest_gray(color: str) -> str:
    # TODO
    pass

# assert farthest_shorthand("#09f166") == "#ff00ff"
# assert nearest_gray("#000306") == "#030303"

### Gợi ý 3 mức

**Bài 1** — (1) Tách `a7`, `c3`, `e0`. (2) Đổi: 167, 195, 224; chia 17 lấy phần dư. (3) `167 = 17·9 + 14` → lên `170 = aa`; `195 = 17·11 + 8` → xuống `187 = bb`; `224 = 17·13 + 3` → xuống `221 = dd`.

**Bài 2** — (1) Đọc hex. (2) Công thức làm tròn. (3) Định dạng `02x`.

**Bài 3** — (1) Mỗi ký tự sau `#` nhân đôi. (2) `"#" + "".join(ch * 2 for ch in short[1:])`. (3) Ngược lại lấy `full[1]`, `full[3]`, `full[5]`.

**Bài 4** — (1) Bình phương lớn nhất khi ở **xa** nhất → luôn là một trong hai đầu `00` hoặc `ff`. (2) `v < 128` → `ff`; ngược lại `00` (`v = 127.5` mới hòa → không xảy ra). (3) Ghép 3 thành phần.

**Bài 5** — (1) Tối thiểu `(R-X)² + (G-X)² + (B-X)²` theo `X`. (2) Đạo hàm bằng 0 ⇒ `X = (R+G+B)/3`. (3) `X` phải nguyên → thử `floor` và `ceil`, chọn tốt hơn (hoặc `round`).

---

## 16. Đáp án tham khảo

In [21]:
answer_ex1_solution = "#aabbdd"

def nearest_component_solution(h):
    v = int(h, 16)
    return f"{17 * ((v + 8) // 17):02x}"

def expand_shorthand_solution(short):
    return "#" + "".join(ch * 2 for ch in short[1:])

def to_shorthand_solution(full):
    return "#" + full[1] + full[3] + full[5]

def farthest_shorthand_solution(color):
    return "#" + "".join("ff" if int(color[i:i + 2], 16) < 128 else "00" for i in range(1, 7, 2))

def nearest_gray_solution(color):
    r, g, b_ = (int(color[i:i + 2], 16) for i in range(1, 7, 2))
    best = min(range(256), key=lambda x: (r - x) ** 2 + (g - x) ** 2 + (b_ - x) ** 2)
    return "#" + f"{best:02x}" * 3

assert answer_ex1_solution == similar_rgb("#a7c3e0")
assert nearest_component_solution("f1") == "ee" and nearest_component_solution("09") == "11"
assert expand_shorthand_solution("#15c") == "#1155cc" and to_shorthand_solution("#1155cc") == "#15c"
assert expand_shorthand_solution(to_shorthand_solution(similar_rgb("#4e3fe1"))) == "#5544dd"
assert farthest_shorthand_solution("#09f166") == "#ff00ff"
# kiểm chứng "xa nhất" bằng brute force 4096 màu
cands = ["#" + f"{17*x:02x}{17*y:02x}{17*z:02x}" for x in range(16) for y in range(16) for z in range(16)]
assert min(cands, key=lambda c: similarity("#09f166", c)) == "#ff00ff"
assert nearest_gray_solution("#000306") == "#030303"
print("Đáp án PASS toàn bộ test!")

Đáp án PASS toàn bộ test!


## 17. Template nhớ nhanh và checklist submit

```python
class Solution:
    def similarRGB(self, color: str) -> str:
        res = "#"
        for i in range(1, 7, 2):
            v = int(color[i:i + 2], 16)
            res += f"{17 * ((v + 8) // 17):02x}"
        return res
```

Công thức nhớ: **viết tắt được ⇔ bội của 17; gần nhất ⇔ `(v + 8) // 17`.**

**Checklist trước khi submit:**
- [ ] Trả chuỗi **7 ký tự** có `#`, chữ thường?
- [ ] Đệm `0` (`02x`) cho thành phần nhỏ như `00`, `11`?
- [ ] Làm tròn **gần nhất**, không phải làm tròn xuống?
- [ ] Xử lý **từng** thành phần riêng (độc lập)?
- [ ] (Premium) Đã đối chiếu đề gốc nếu có tài khoản?